# Artificial and Computational Intelligence Assignment 1

## Problem solving by Uninformed & Informed Search

List all the team members BITS ID ,Name along with % of contribution in this assignment: sample Provided below:
1. 2024TM93056 - Mallidi Akhil Reddy - 100%
2. 2024TM93057 - Ashwarya Anupam - 100%
3. 2024TM93058 - Vaidya Rucha Sandeep - 100%
4. 2024TM93059 - Shivam Prabhakar - 100%
5. 2024TM93061 - J Rakesh - 100%

Things to follow
1.	Use appropriate data structures to represent the graph and the path using python libraries
2.	Provide proper documentation
3.	Find the path and print it

Coding begins here

In [ ]:
# Importing the required librraries

import heapq
import time
import sys
from typing import List, Tuple, Optional, Set, Dict
from collections import deque



```
# This is formatted as code
```

### 1.	Define the environment in the following block

List the PEAS decription of the problem here in this markdown block


Performance Measure:
- Successfully reach home from office
- Minimize path length (number of squares)
- Maximize safety points (avoid proximity to obstacles)
- Path cost: Lower is better (safety points subtracted from base cost)

Environment:
- Type: Static, Fully Observable, Deterministic, Discrete, Sequential
- Grid-based city map with buildings and roadblocks
- No diagonal movements allowed
- Buildings and roadblocks are impassable

Actuators:
- Move Up (North)
- Move Down (South)
- Move Left (West)
- Move Right (East)

Sensors:
- Position sensor (current location)
- Obstacle detector (buildings, roadblocks)
- Proximity sensor (adjacent cells for safety calculation)
- Goal detector (home location)


Design the agent as PSA Agent(Problem Solving Agent)
Clear Initial data structures to define the graph and variable declarations is expected
IMPORTATANT: Write distinct code block as below

In [ ]:
#Code Block : Set Initial State (Must handle dynamic inputs)

class GridEnvironment:

    # Defining cell type constants

    EMPTY = 0
    BUILDING = 1
    ROADBLOCK = 2
    OFFICE = 3
    HOME = 4

    def __init__(self, rows: int, cols: int):
        """ Initializing grid environment """

        self.rows = rows
        self.cols = cols
        self.grid = [[self.EMPTY for _ in range(cols)] for _ in range(rows)]
        self.start = None
        self.goal = None

    def set_cell(self, row: int, col: int, cell_type: int):
        """ Set cell type in grid """

        if 0 <= row < self.rows and 0 <= col < self.cols:
            self.grid[row][col] = cell_type

    def set_start(self, row: int, col: int):
        """ Set starting position (Office) """

        self.start = (row, col)
        self.set_cell(row, col, self.OFFICE)

    def set_goal(self, row: int, col: int):
        """ Set goal position (Home) """

        self.goal = (row, col)
        self.set_cell(row, col, self.HOME)

    def add_building(self, row: int, col: int):
        """ Add building obstacle """

        self.set_cell(row, col, self.BUILDING)

    def add_roadblock(self, row: int, col: int):
        """ Add roadblock obstacle """

        self.set_cell(row, col, self.ROADBLOCK)

    def display_grid(self):
        """ Display the grid with legend """

        symbols = {
            self.EMPTY: '.',
            self.BUILDING: 'B',
            self.ROADBLOCK: 'X',
            self.OFFICE: 'S',
            self.HOME: 'G'
        }

        print("\n" + "="*60)
        print("GRID MAP")
        print("="*60)
        print("Legend: S=Start(Office), G=Goal(Home), B=Building, X=Roadblock, .=Empty")
        print("\n   ", end="")
        for col in range(self.cols):
            print(f"{col:2}", end=" ")
        print()

        for row in range(self.rows):
            print(f"{row:2} ", end=" ")
            for col in range(self.cols):
                print(f"{symbols[self.grid[row][col]]:2}", end=" ")
            print()
        print()


In [ ]:
#Code Block : Set the matrix for transition & cost (as relevant for the given problem)

class TransitionModel:
    """Handles state transitions and cost calculations"""

    def __init__(self, environment: GridEnvironment):
        self.env = environment
        # Possible moves: Up, Down, Left, Right (no diagonals)
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.action_names = ['UP', 'DOWN', 'LEFT', 'RIGHT']

    def is_valid_position(self, row: int, col: int) -> bool:
        """Check if position is within bounds and traversable"""
        if row < 0 or row >= self.env.rows or col < 0 or col >= self.env.cols:
            return False
        cell = self.env.grid[row][col]
        # Can't pass through buildings or roadblocks
        return cell not in [self.env.BUILDING, self.env.ROADBLOCK]

    def calculate_safety_points(self, row: int, col: int) -> int:
        """
        Calculate safety points for a position
        +5 points when adjacent to buildings
        +3 points when adjacent to roadblocks
        Higher points = more penalty (less safe)
        """
        safety_points = 0

        # Check all 4 adjacent cells (no diagonals)
        adjacent = [(row-1, col), (row+1, col), (row, col-1), (row, col+1)]

        for adj_row, adj_col in adjacent:
            if 0 <= adj_row < self.env.rows and 0 <= adj_col < self.env.cols:
                cell = self.env.grid[adj_row][adj_col]
                if cell == self.env.BUILDING:
                    safety_points += 5
                elif cell == self.env.ROADBLOCK:
                    safety_points += 3

        return safety_points

    def get_path_cost(self, row: int, col: int) -> float:
        """
        Calculate step cost for moving to this position
        Base cost = 1 (for moving one square)
        Add safety penalty points
        """
        base_cost = 1
        safety_penalty = self.calculate_safety_points(row, col)
        return base_cost + safety_penalty

In [ ]:
#Code Block : Write function to design the Transition Model/Successor function. Ideally this would be called while search algorithms are implemented

def get_successors(state: Tuple[int, int], transition_model: TransitionModel) -> List[Tuple[Tuple[int, int], float, str]]:
    """
    Generate successor states from current state
    Returns: List of (next_state, cost, action) tuples
    """
    successors = []
    row, col = state

    for i, (dr, dc) in enumerate(transition_model.actions):
        new_row, new_col = row + dr, col + dc

        if transition_model.is_valid_position(new_row, new_col):
            next_state = (new_row, new_col)
            cost = transition_model.get_path_cost(new_row, new_col)
            action = transition_model.action_names[i]
            successors.append((next_state, cost, action))

    return successors

In [ ]:
#Code block : Write fucntion to handle goal test (Must handle dynamic inputs). Ideally this would be called while search algorithms are implemented

def is_goal(state: Tuple[int, int], goal: Tuple[int, int]) -> bool:
    """
    Check if current state is the goal state
    Handles dynamic goal input
    """
    return state == goal

### 2.	Definition of Algorithm 1 (Recursive Best First Search Algorithm)

In [ ]:
#Code Block : Function for algorithm 1 implementation

class Node:
    """ Node class for search tree """

    def __init__(self, state: Tuple[int, int], parent=None, action: str = None,
                 g_cost: float = 0, h_cost: float = 0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g_cost = g_cost  # Cost from start to current
        self.h_cost = h_cost  # Heuristic cost to goal
        self.f_cost = g_cost + h_cost  # Total estimated cost

    def __lt__(self, other):
        return self.f_cost < other.f_cost

    def get_path(self) -> List[Tuple[int, int]]:
        """ Reconstruct path from start to this node """
        path = []
        node = self
        while node is not None:
            path.append(node.state)
            node = node.parent
        return path[::-1]

    def get_actions(self) -> List[str]:
        """ Get sequence of actions to reach this node """
        actions = []
        node = self
        while node.parent is not None:
            actions.append(node.action)
            node = node.parent
        return actions[::-1]

def manhattan_distance(state: Tuple[int, int], goal: Tuple[int, int]) -> int:
    """ Calculate Manhattan distance heuristic (admissible) """
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])


In [ ]:
#RBFS Implementation


class RBFSSearch:
    """ Recursive Best First Search Algorithm """

    def __init__(self, environment: GridEnvironment, transition_model: TransitionModel):
        self.env = environment
        self.transition = transition_model
        self.nodes_expanded = 0
        self.max_frontier_size = 0
        self.start_time = 0
        self.end_time = 0

    def rbfs_recursive(self, node: Node, f_limit: float) -> Tuple[Optional[Node], float]:
        """
        Recursive Best First Search - Core Algorithm
        Returns: (solution_node, new_f_limit)
        """
        self.nodes_expanded += 1

        # Goal test
        if is_goal(node.state, self.env.goal):
            return node, None

        # Generate successors
        successors = []
        for next_state, step_cost, action in get_successors(node.state, self.transition):
            g_cost = node.g_cost + step_cost
            h_cost = manhattan_distance(next_state, self.env.goal)
            child = Node(next_state, node, action, g_cost, h_cost)
            # Update f-cost to be at least parent's f-cost
            child.f_cost = max(child.g_cost + child.h_cost, node.f_cost)
            successors.append(child)

        if not successors:
            return None, float('inf')

        # Track maximum frontier size
        self.max_frontier_size = max(self.max_frontier_size, len(successors))

        while True:
            # Sort successors by f_cost (best first)
            successors.sort(key=lambda x: x.f_cost)

            best = successors[0]

            # If best f-cost exceeds limit, return failure
            if best.f_cost > f_limit:
                return None, best.f_cost

            # Get alternative f-cost (second best)
            alternative = successors[1].f_cost if len(successors) > 1 else float('inf')

            # Recursive call with updated limit
            result, best.f_cost = self.rbfs_recursive(best, min(f_limit, alternative))

            if result is not None:
                return result, None

    def search(self) -> Optional[Node]:
        """ Execute RBFS search """
        self.start_time = time.time()
        start_node = Node(self.env.start, None, None, 0,
                         manhattan_distance(self.env.start, self.env.goal))

        result, _ = self.rbfs_recursive(start_node, float('inf'))
        self.end_time = time.time()

        return result


### 3.	Definition of Algorithm 2 (Uniform Cost Search)

In [ ]:
# Code Block : Function for algorithm 2 implementation (UCS)

class UCSSearch:
    """ Uniform Cost Search Algorithm """

    def __init__(self, environment: GridEnvironment, transition_model: TransitionModel):
        self.env = environment
        self.transition = transition_model
        self.nodes_expanded = 0
        self.max_frontier_size = 0
        self.start_time = 0
        self.end_time = 0

    def search(self) -> Optional[Node]:
        """ Execute UCS search """
        self.start_time = time.time()

        # Start node
        start_node = Node(self.env.start, None, None, g_cost=0, h_cost=0)

        # Priority queue ordered by g_cost (path cost so far)
        frontier = []
        heapq.heappush(frontier, (start_node.g_cost, start_node))

        explored = set()

        while frontier:
            cost, node = heapq.heappop(frontier)
            self.nodes_expanded += 1
            self.max_frontier_size = max(self.max_frontier_size, len(frontier))

            # Goal test
            if node.state == self.env.goal:
                self.end_time = time.time()
                return node

            if node.state in explored:
                continue
            explored.add(node.state)

            # Generate successors
            for next_state, step_cost, action in get_successors(node.state, self.transition):
                if next_state not in explored:
                    g_cost = node.g_cost + step_cost
                    child = Node(next_state, node, action, g_cost, 0)  # h=0 in UCS
                    heapq.heappush(frontier, (child.g_cost, child))

        self.end_time = time.time()
        return None


### DYNAMIC INPUT

IMPORTANT : Dynamic Input must be got in this section. Display the possible states to choose from:
This is applicable for all the relevent problems as mentioned in the question.

In [ ]:
def create_environment_from_image():
    """
    Create environment based on the provided image
    Grid is 6x6 as shown in the problem statement
    """
    env = GridEnvironment(6, 6)

    # Set start (Office) at position (0, 0)
    env.set_start(0, 0)

    # Set goal (Home) at position (5, 5)
    env.set_goal(5, 5)

    # Add buildings from the image
    buildings = [
        (1, 0),  # Hospital
        (3, 0),  # Meeting Room
        (3, 5),  # Grid building
        (5, 1)   # Meeting room
    ]

    for row, col in buildings:
        env.add_building(row, col)

    # Add roadblocks from the image
    roadblocks = [
        (0, 2),
        (0, 3),
        (1, 2),
        (2, 4),
        (3, 2),
        (5, 3)
    ]

    for row, col in roadblocks:
        env.add_roadblock(row, col)

    return env


def get_user_input():
    """Get dynamic input for start and goal positions"""
    print("\n" + "="*60)
    print("DYNAMIC INPUT SECTION")
    print("="*60)

    print("\nEnter coordinates as (row, col) where both are between 0-5")
    print("Example: For position (1,1) enter: 1 1")

    # Get start position
    while True:
        try:
            start_input = input("\nEnter START position (Office) [row col]: ").strip()
            start_row, start_col = map(int, start_input.split())
            if 0 <= start_row <= 5 and 0 <= start_col <= 5:
                break
            print("Invalid input! Coordinates must be between 0 and 5.")
        except:
            print("Invalid input! Please enter two numbers separated by space.")

    # Get goal position
    while True:
        try:
            goal_input = input("Enter GOAL position (Home) [row col]: ").strip()
            goal_row, goal_col = map(int, goal_input.split())
            if 0 <= goal_row <= 5 and 0 <= goal_col <= 5:
                break
            print("Invalid input! Coordinates must be between 0 and 5.")
        except:
            print("Invalid input! Please enter two numbers separated by space.")

    return (start_row, start_col), (goal_row, goal_col)


### 4.	Calling the search algorithms
(For bidirectional search in below sections first part can be used as per Hint provided. Under second section other combinations as per Hint or your choice of 2 algorithms can be called .As an analyst suggest suitable approximation in the comparitive analysis section)

In [ ]:
#Invoke algorithm 1 (Should Print the solution, path, cost etc., (As mentioned in the problem))

def print_solution(algorithm_name: str, result: Optional[Node], search_obj):
    """ Print solution details """
    print("\n" + "="*60)
    print(f"{algorithm_name} - RESULTS")
    print("="*60)

    if result is None:
        print("\nNo path found!")
        return

    path = result.get_path()
    actions = result.get_actions()

    print(f"\n✓ PATH FOUND!")
    print(f"\nPath: {path}")
    print(f"Actions: {actions}")
    print(f"\nNumber of squares in path: {len(path)}")
    print(f"Total path cost: {result.g_cost:.2f}")

    # Calculate total safety points
    transition = search_obj.transition
    total_safety_points = sum(transition.calculate_safety_points(r, c) for r, c in path)
    print(f" Total safety penalty points: {total_safety_points}")

    # Display path on grid
    print("\n" + "-"*60)
    print("PATH VISUALIZATION")
    print("-"*60)

    grid_copy = [row[:] for row in search_obj.env.grid]
    for i, (row, col) in enumerate(path):
        if grid_copy[row][col] not in [search_obj.env.OFFICE, search_obj.env.HOME]:
            grid_copy[row][col] = 5  # Path marker

    symbols = {0: '.', 1: 'B', 2: 'X', 3: 'S', 4: 'G', 5: '*'}

    print("\n   ", end="")
    for col in range(search_obj.env.cols):
        print(f"{col:2}", end=" ")
    print()

    for row in range(len(grid_copy)):
        print(f"{row:2} ", end=" ")
        for col in range(len(grid_copy[0])):
            print(f"{symbols[grid_copy[row][col]]:2}", end=" ")
        print()
    print("\nLegend: S=Start, G=Goal, B=Building, X=Roadblock, *=Path, .=Empty\n")


def run_rbfs(env, transition):
    """ Execute RBFS algorithm """
    print("\n" + "="*60)
    print("EXECUTING ALGORITHM 1: RECURSIVE BEST FIRST SEARCH (RBFS)")
    print("="*60)

    rbfs = RBFSSearch(env, transition)
    result = rbfs.search()
    print_solution("RBFS", result, rbfs)

    return rbfs, result


In [ ]:
#Invoke algorithm 2 (Should Print the solution, path, cost etc., (As mentioned in the problem))
def run_ucs(env, transition):
    """Execute Uniform Cost Search"""
    print("\n" + "="*60)
    print("EXECUTING ALGORITHM 2: UNIFORM COST SEARCH (UCS)")
    print("="*60)

    ucs = UCSSearch(env, transition)
    result = ucs.search()
    print_solution("UCS", result, ucs)

    return ucs, result

### 5.	Comparitive Analysis (Time and Space Complexity)

In [ ]:
#Code Block : Print the Time & Space complexity of algorithm 1

def print_rbfs_complexity(rbfs: RBFSSearch, result: Optional[Node]):
    """ Print time and space complexity for RBFS """

    print("\n" + "="*60)
    print("COMPLEXITY ANALYSIS - RBFS")
    print("="*60)

    print(f"\nNodes Expanded: {rbfs.nodes_expanded}")
    print(f"Maximum Frontier Size: {rbfs.max_frontier_size}")
    print(f"Execution Time: {(rbfs.end_time - rbfs.start_time)*1000:.4f} ms")

    if result:
        print(f"Solution Depth: {len(result.get_path()) - 1}")

    print("\n" + "-"*60)
    print("THEORETICAL COMPLEXITY")
    print("-"*60)
    print("Time Complexity: O(b^d)")
    print("  where b = branching factor (max 4 in this grid)")
    print("  where d = depth of optimal solution")
    print(f"\nActual branching factor: ~{rbfs.nodes_expanded / max(1, len(result.get_path()) if result else 1):.2f}")

    print("\nSpace Complexity: O(bd)")
    print("  Linear space due to recursive stack")
    print(f"  Actual maximum frontier size: {rbfs.max_frontier_size} nodes")

    # Memory estimate
    node_size = sys.getsizeof(Node((0,0)))
    estimated_memory = rbfs.max_frontier_size * node_size
    print(f"  Estimated memory usage: ~{estimated_memory/1024:.2f} KB")

In [ ]:
#Code Block : Print the Time & Space complexity of algorithm 2

def print_ucs_complexity(ucs: UCSSearch, result: Optional[Node]):
    """ Print time and space complexity for UCS """

    print("\n" + "="*60)
    print("COMPLEXITY ANALYSIS - UCS")
    print("="*60)

    print(f"\nNodes Expanded: {ucs.nodes_expanded}")
    print(f"Maximum Frontier Size: {ucs.max_frontier_size}")
    print(f"Execution Time: {(ucs.end_time - ucs.start_time)*1000:.4f} ms")

    if result:
        print(f"Solution Depth: {len(result.get_path()) - 1}")

    print("\n" + "-"*60)
    print("THEORETICAL COMPLEXITY")
    print("-"*60)
    print("Time Complexity: O(b^d)")
    print("  where b = branching factor (max 4 in this grid)")
    print("  where d = depth of optimal solution (measured in cost, not steps)")
    print(f"\nActual branching factor: ~{ucs.nodes_expanded / max(1, len(result.get_path()) if result else 1):.2f}")

    print("\nSpace Complexity: O(b^d)")
    print("  UCS uses a priority queue (frontier) that can store many nodes")
    print(f"  Actual maximum frontier size: {ucs.max_frontier_size} nodes")

    # Memory estimate
    node_size = sys.getsizeof(Node((0,0)))
    estimated_memory = ucs.max_frontier_size * node_size
    print(f"  Estimated memory usage: ~{estimated_memory/1024:.2f} KB")


MAIN EXECUTION

In [ ]:
def main():
    """Main execution function"""
    print("\n" + "="*60)
    print("GPS NAVIGATION AGENT - ACI ASSIGNMENT 1")
    print("="*60)
    print("\nAlgorithms:")
    print("1. Recursive Best First Search (RBFS)")
    print("2. Uniform Cost Search (UCS)")

    # Create environment from image
    env = create_environment_from_image()

    # Display initial grid
    env.display_grid()

    # Option for dynamic input
    print("\nDo you want to use default positions or enter custom positions?")
    choice = input("Enter 'D' for default or 'C' for custom: ").strip().upper()

    if choice == 'C':
        start, goal = get_user_input()
        env.set_start(start[0], start[1])
        env.set_goal(goal[0], goal[1])
        env.display_grid()

    # Create transition model
    transition = TransitionModel(env)

    # -------------------------------
    # Run RBFS
    # -------------------------------
    rbfs, rbfs_result = run_rbfs(env, transition)
    print_rbfs_complexity(rbfs, rbfs_result)

    # -------------------------------
    # Run UCS
    # -------------------------------
    ucs, ucs_result = run_ucs(env, transition)

    print("\n" + "="*60)
    print("COMPARATIVE ANALYSIS")
    print("="*60)

    def algo_stats(name, search_obj, result):
        if result is None:
            return {
                "Path Length": "-",
                "Path Cost": "-",
                "Nodes Expanded": "-",
                "Max Frontier": "-",
                "Time (ms)": "-"
            }
        return {
            "Path Length": len(result.get_path()),
            "Path Cost": f"{result.g_cost:.2f}",
            "Nodes Expanded": getattr(search_obj, "nodes_expanded", "-"),
            "Max Frontier": getattr(search_obj, "max_frontier_size", "-"),
            "Time (ms)": f"{(search_obj.end_time - search_obj.start_time)*1000:.4f}"
                         if hasattr(search_obj, "end_time") else "-"
        }

    rbfs_stats = algo_stats("RBFS", rbfs, rbfs_result)
    ucs_stats = algo_stats("UCS", ucs, ucs_result)

    print(f"\n{'Metric':<20}{'RBFS':<15}{'UCS':<15}")
    print("-"*50)
    for key in rbfs_stats.keys():
        print(f"{key:<20}{rbfs_stats[key]:<15}{ucs_stats[key]:<15}")


if __name__ == "__main__":
    main()



GPS NAVIGATION AGENT - ACI ASSIGNMENT 1

Algorithms:
1. Recursive Best First Search (RBFS)
2. Uniform Cost Search (UCS)

GRID MAP
Legend: S=Start(Office), G=Goal(Home), B=Building, X=Roadblock, .=Empty

    0  1  2  3  4  5 
 0  S  .  X  X  .  .  
 1  B  .  X  .  .  .  
 2  .  .  .  .  X  .  
 3  B  .  X  .  .  B  
 4  .  .  .  .  .  .  
 5  .  B  .  X  .  G  


Do you want to use default positions or enter custom positions?
Enter 'D' for default or 'C' for custom: D

EXECUTING ALGORITHM 1: RECURSIVE BEST FIRST SEARCH (RBFS)

RBFS - RESULTS

✓ PATH FOUND!

Path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (3, 3), (4, 3), (4, 4), (5, 4), (5, 5)]
Actions: ['RIGHT', 'DOWN', 'DOWN', 'RIGHT', 'RIGHT', 'DOWN', 'DOWN', 'RIGHT', 'DOWN', 'RIGHT']

Number of squares in path: 11
Total path cost: 39.00
 Total safety penalty points: 34

------------------------------------------------------------
PATH VISUALIZATION
------------------------------------------------------------

    0  1  2  3  

### 6.	Provide your comparitive analysis or findings in no more than 3 lines in below section

Comparison :

*   Both RBFS and UCS found the same optimal path with equal cost and path length.
*   However, UCS expanded fewer nodes and executed faster, showing better efficiency in this grid.
*   RBFS used less memory (smaller frontier) due to its recursive nature but incurred higher time overhead.
